In [ ]:
# Required Imports & Session Setup
import os
import uuid
import logging
from datetime import datetime
import requests

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, FloatType, NullType
)

from typing import List, Tuple, Dict, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Configure logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)
logger.info("Imports and session setup complete.")

### Define dictionary for Source and Target database details for benchmark testing. This Dictionary can be executed multiple times, and if track details already exists then it bypass the execution.

In [ ]:
# Dictionary for entering Power BI report details into metadata.tbl_LoadTestConnectionDetailsDAX table
connectionDetails = [
    {
        "TrackName": "Track1",
        "SourceServerName": "server1.database.windows.net",
        "SourceDatabaseName": "Database1",
        "TargetServerName": "server2.msit-datawarehouse.fabric.microsoft.com",
        "TargetDatabaseName": "Database1",
        "isActive": True
    },
    {
        "TrackName": "Track2",
        "SourceServerName": "server1.database.windows.net",
        "SourceDatabaseName": "Database2",
        "TargetServerName": "server2.msit-datawarehouse.fabric.microsoft.com",
        "TargetDatabaseName": "Database2",
        "isActive": True
    }
]

### Metadata and Logging table scripts, these scripts takes care of existing DB objects and if it already exists then it won't create new.

In [ ]:
ddl_statements = [
    # SCHEMAS
    """
    IF NOT EXISTS (
        SELECT 1
        FROM sys.schemas
        WHERE name = 'metadata'
    )
    BEGIN
        EXEC('CREATE SCHEMA [metadata]')
    END
    """,
    """
    IF NOT EXISTS (
        SELECT 1
        FROM sys.schemas
        WHERE name = 'logging'
    )
    BEGIN
        EXEC('CREATE SCHEMA [logging]')
    END
    """,
    # TABLE: metadata.SPT_ConnectionDetails
    """
    IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'metadata'
        AND t.name = 'SPT_ConnectionDetails'
    )
    BEGIN
        CREATE TABLE [metadata].[SPT_ConnectionDetails](
            [ID] [int] IDENTITY(1,1) NOT NULL,
            [TrackName] [nvarchar](500) NULL,
            [SourceServerName] [nvarchar](500) NULL,
            [SourceDatabaseName] [nvarchar](500) NULL,
            [TargetServerName] [nvarchar](256) NULL,
            [TargetDatabaseName] [nvarchar](256) NULL,
            [IsActive] [int] NOT NULL,
            [CreatedOn] [datetime] NULL
        ) 
        ALTER TABLE [metadata].[SPT_ConnectionDetails] ADD  DEFAULT (getdate()) FOR [CreatedOn]
    END
    """,
   # [metadata].[SPT_QueryCollection]
    """IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'metadata'
        AND t.name = 'SPT_QueryCollection'
    )
    BEGIN
        CREATE TABLE [metadata].[SPT_QueryCollection](
            [QueryId] [int] IDENTITY(1,1) NOT NULL,
            [TrackName] [nvarchar](500) NOT NULL,
            [QueryText] [nvarchar](max) NOT NULL,
            [MaxConcurrency] [int] NULL,
            [IsActive] [int] NOT NULL,
            [CreatedOn] [datetime] NULL,
            [QueryUser] [varchar](255) NULL
        ) 
        ALTER TABLE [metadata].[SPT_QueryCollection] ADD  DEFAULT ((5)) FOR [MaxConcurrency]
        ALTER TABLE [metadata].[SPT_QueryCollection] ADD  DEFAULT (getdate()) FOR [CreatedOn]
    END
    """,
    #[logging].[SPT_QueryTestLog]
    """
        IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'logging'
        AND t.name = 'SPT_QueryTestLog'
    )
    BEGIN
        CREATE TABLE [logging].[SPT_QueryTestLog]
        (
            [RunId] [nvarchar](max) NOT NULL,
            [TrackName] [nvarchar](max) NULL,
            [Concurrency] [int] NOT NULL,
            [QueryId] [nvarchar](max) NULL,
            [QueryText] [nvarchar](max) NULL,
            [AzSQL_ExecutionTime] [float] NULL,
            [AzSQL_StartTime] [datetime] NULL,
            [AzSQL_EndTime] [datetime] NULL,
            [Fabric_ExecutionTime] [float] NULL,
            [Fabric_StartTime] [datetime] NULL,
            [Fabric_EndTime] [datetime] NULL
        ) 
    END
    """,
    #========== VIEW ==========
    """
    IF OBJECT_ID('metadata.vwSPT_QueryTestConfig', 'V') IS NOT NULL
        DROP VIEW [metadata].[vwSPT_QueryTestConfig]
    """,
    """
    CREATE VIEW [metadata].[vwSPT_QueryTestConfig]
    AS
    SELECT A.QueryId
        ,A.TrackName
        ,A.QueryText
        ,A.MaxConcurrency
        ,B.SourceServerName
        ,B.SourceDatabaseName
        ,B.TargetServerName
        ,B.TargetDatabaseName
        ,A.isActive
    FROM metadata.SPT_QueryCollection A
    JOIN metadata.SPT_ConnectionDetails B ON A.TrackName = B.TrackName
    """
]

In [ ]:
#Escapes single quotes in SQL string literals to prevent syntax errors or SQL injection issues.
def _escape_sql_literal(val: str) -> str:
    """Escape single quotes within SQL string literals by doubling them."""
    if val is None:
        return ""
    return str(val).replace("'", "''")

In [ ]:
def get_access_token(resource: str = "https://database.windows.net/", name: str = "") -> str:
    """
    Fetch an AAD access token for the given resource (audience).
    Uses mssparkutils.credentials.getToken under the hood.
    """
    return mssparkutils.credentials.getToken(resource, name)

In [ ]:
def execute_sql_statements_with_token(
    sql_statements: List[str],
    jdbc_url: str,
    access_token: Optional[str] = None
) -> List[Dict[str, Optional[str]]]:
    """
    Execute one or more raw T-SQL statements via JDBC using an access token.
    Returns a list of dictionaries for each statement with keys:
      - "sql": the statement executed
      - "status": "success" or "failed"
      - "error": error message if failed, else None
    """
    results: List[Dict[str, Optional[str]]] = []
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    conn = None
    try:
        # Setup connection properties
        props = spark._sc._gateway.jvm.java.util.Properties()
        props.setProperty("accessToken", access_token)
        props.setProperty("encrypt", "true")

        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
        conn = driver_manager.getConnection(jdbc_url, props)

        for sql in sql_statements:
            try:
                stmt = conn.createStatement()
                stmt.execute(sql)
                results.append({"sql": sql, "status": "success", "error": None})
            except Exception as ex_stmt:
                results.append({"sql": sql, "status": "failed", "error": str(ex_stmt)})
            finally:
                try:
                    stmt.close()
                except Exception:
                    # Log or ignore close errors
                    pass

    except Exception as ex_conn:
        # If connection establishment fails
        err_msg = f"Connection setup failed: {ex_conn}"
        # Append an entry for each statement? Or a single global error?
        results.append({"sql": None, "status": "failed", "error": err_msg})
    finally:
        if conn:
            try:
                conn.close()
            except Exception:
                # Log or ignore
                pass

    return results

In [ ]:
def read_sql_data(query: str, jdbc_url: str, access_token: str) -> pd.DataFrame:
    """Execute SQL query via JDBC and return as Pandas DataFrame."""
    props = spark._sc._gateway.jvm.java.util.Properties()
    props.setProperty("accessToken", access_token)
    props.setProperty("encrypt", "true")
    driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager

    con = driver_manager.getConnection(jdbc_url, props)
    stmt = con.prepareCall(query)

    data = []
    try:
        stmt.execute()
        result_set = stmt.getResultSet()
        if result_set:
            while result_set.next():
                row = {
                    result_set.getMetaData().getColumnName(i): result_set.getString(i)
                    for i in range(1, result_set.getMetaData().getColumnCount() + 1)
                }
                data.append(row)
    finally:
        stmt.close()
        con.close()

    return pd.DataFrame(data)

In [ ]:
def write_sql_data(df, table_name: str, jdbc_url: str, access_token: str) -> None:
    """Write Spark or Pandas DataFrame to SQL table."""
    spark_df = spark.createDataFrame(df) if isinstance(df, pd.DataFrame) else df
    spark_df.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", table_name) \
        .option("accessToken", access_token) \
        .option("encrypt", "true") \
        .mode("append") \
        .save()

In [ ]:
def execute_sql_queries_concurrently(df, jdbc_url: str, concurrency: int = 5):
    """Execute SQL queries concurrently against a given JDBC endpoint."""
    access_token = get_access_token()
    environment = "Azure SQL" if "database.windows.net" in jdbc_url else "Fabric"

    def run_query(query_id, trackName, query_text) -> Tuple:
        try:
            props = spark._sc._gateway.jvm.java.util.Properties()
            props.setProperty("accessToken", access_token)
            props.setProperty("encrypt", "true")
            props.setProperty("trustServerCertificate", "true")
            driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager

            start_time = datetime.now()
            conn = driver_manager.getConnection(jdbc_url, props)
            stmt = conn.createStatement()
            stmt.execute(query_text)
            stmt.close()
            conn.close()
            end_time = datetime.now()

            return (query_id, trackName, query_text, "Success",
                    start_time, end_time, (end_time - start_time).total_seconds())
        except Exception as e:
            logging.error("Error running query %s: %s", query_id, str(e))
            return (query_id, trackName, query_text, f"Error: {str(e)}",
                    None, None, None)

    query_pandas_df = df.select("QueryId", "TrackName", "QueryText").toPandas()
    results = []

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [
            executor.submit(run_query, row["QueryId"], row["TrackName"], row["QueryText"])
            for _, row in query_pandas_df.iterrows()
        ]
        for future in as_completed(futures):
            results.append(future.result())

    final_result_df = spark.createDataFrame(
        results,
        schema=["QueryId", "TrackName", "QueryText", "Status",
                "StartTime", "EndTime", "ExecutionTime"]
    )

    return (
        final_result_df
        .withColumn("Environment", F.lit(environment))
        .withColumn("Concurrency", F.lit(concurrency))
    )

In [ ]:
def main():
    config_jdbc_url = f"jdbc:sqlserver://{server};databaseName={database};"

    # Get metadata
    config_query = f"""
        SELECT DISTINCT sourceServerName,sourceDatabaseName,
                        targetServerName,targetDatabaseName
        FROM metadata.vwSPT_QueryTestConfig 
        WHERE TrackName = '{trackName}'
    """
    config_df = read_sql_data(config_query, config_jdbc_url, get_access_token())
    spark_df = spark.createDataFrame(config_df)

    src_server = spark_df.first()["sourceServerName"]
    src_db = spark_df.first()["sourceDatabaseName"]
    tgt_server = spark_df.first()["targetServerName"]
    tgt_db = spark_df.first()["targetDatabaseName"]

    prod_jdbc_url = f"jdbc:sqlserver://{src_server}:1433;database={src_db}"
    fabric_jdbc_url = f"jdbc:sqlserver://{tgt_server}:1433;database={tgt_db}"

    # Get queries
    query_df = read_sql_data(
        f"SELECT QueryId, TrackName, QueryText, MaxConcurrency "
        f"FROM metadata.vwSPT_QueryTestConfig WHERE isActive=1 AND TrackName='{trackName}'",
        config_jdbc_url,
        get_access_token()
    )
    query_spark_df = spark.createDataFrame(query_df)

    # Execute
    result_azsql = execute_sql_queries_concurrently(query_spark_df, prod_jdbc_url, concurrency=5)
    result_fabric = execute_sql_queries_concurrently(query_spark_df, fabric_jdbc_url, concurrency=5)

    runtime_id = str(uuid.uuid4())
    result_azsql.createOrReplaceTempView("AzSQL_results")
    result_fabric.createOrReplaceTempView("Fabric_results")
    query_spark_df.createOrReplaceTempView("Query_df")

    final_result = spark.sql(f"""
        SELECT '{runtime_id}' as RunId,
               A.TrackName,
               A.Concurrency,
               A.QueryId,
               A.QueryText, 
               ROUND(A.ExecutionTime,2) as AzSQL_ExecutionTime,
               A.StartTime as AzSQL_StartTime,
               A.EndTime as AzSQL_EndTime, 
               ROUND(B.ExecutionTime,2) as Fabric_ExecutionTime,
               B.StartTime as Fabric_StartTime,
               B.EndTime as Fabric_EndTime
        FROM AzSQL_results A
        LEFT JOIN Fabric_results B 
               ON A.QueryId = B.QueryId AND A.TrackName = B.TrackName
        LEFT JOIN Query_df C ON A.QueryId = C.QueryId 
    """)

    display(final_result)
    
    # capture the execution details into log table
    write_sql_data(final_result, "logging.SPT_QueryTestLog", config_jdbc_url, get_access_token())
    logging.info("Execution completed successfully.")

In [ ]:
def batch_insert_conn_if_not_exists(conn_list: list[dict], jdbc_url: str, exec_fn) -> list[dict]:
    """
    Loop over a list of connection dicts, build & execute SQL with “NOT EXISTS” logic.
    exec_fn(sql: str) => executes the SQL (e.g. via JDBC or your token-based execution).
    Returns a list of result dicts for each attempted insert.
    """
    results = []
    for conn in conn_list:
        try:
            sql = build_insert_if_not_exists_sql(conn)
        except Exception as e:
            results.append({
                "conn": conn,
                "status": "failed",
                "error": f"SQL build error: {e}"
            })
            continue

        try:
            resp = exec_fn(sql)
            results.append({
                "conn": conn,
                "status": "success",
                "response": resp
            })
        except Exception as ex:
            results.append({
                "conn": conn,
                "status": "failed",
                "error": str(ex),
                "sql": sql
            })
    return results

In [ ]:
def build_insert_if_not_exists_sql(conn: dict) -> str:
    """
    Given a connection-details dict, build a SQL INSERT … WHERE NOT EXISTS statement.
    """
    required = [
        "TrackName", "SourceServerName",
        "SourceDatabaseName", "TargetServerName",
        "TargetDatabaseName", "isActive"
    ]

    missing = [k for k in required if k not in conn]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")

    try:
        # Escape each string field using your helper
        tn  = _escape_sql_literal(conn["TrackName"])
        ssn = _escape_sql_literal(conn["SourceServerName"])
        sdn = _escape_sql_literal(conn["SourceDatabaseName"])
        tsn = _escape_sql_literal(conn["TargetServerName"])
        tdn = _escape_sql_literal(conn["TargetDatabaseName"])
        is_act = 1 if conn["isActive"] else 0

        sql = f"""
        INSERT INTO metadata.SPT_ConnectionDetails
            (TrackName,
             SourceServerName,
             SourceDatabaseName,
             TargetServerName,
             TargetDatabaseName,
             isActive)
        SELECT
            '{tn}'  AS TrackName,
            '{ssn}' AS SourceServerName,
            '{sdn}' AS SourceDatabaseName,
            '{tsn}' AS TargetServerName,
            '{tdn}' AS TargetDatabaseName,
            {is_act} AS isActive
        WHERE NOT EXISTS (
            SELECT 1
            FROM metadata.SPT_ConnectionDetails AS t
            WHERE LOWER(t.TrackName) = LOWER('{tn}')
        );
        """
        return sql

    except Exception as e:
        raise RuntimeError(f"Failed to build SQL for connection: {conn}. Error: {e}") from e


In [ ]:
def execute_sql_statements_with_token(
    sql_statements: List[str],
    jdbc_url: str,
    access_token: Optional[str] = None
) -> List[Dict[str, Optional[str]]]:
    """
    Execute one or more raw T-SQL statements via JDBC using an access token.
    Returns a list of dictionaries for each statement with keys:
      - "sql": the statement executed
      - "status": "success" or "failed"
      - "error": error message if failed, else None
    """
    results: List[Dict[str, Optional[str]]] = []
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")
    conn = None
    try:
        # Setup connection properties
        props = spark._sc._gateway.jvm.java.util.Properties()
        props.setProperty("accessToken", access_token)
        props.setProperty("encrypt", "true")

        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
        conn = driver_manager.getConnection(jdbc_url, props)

        for sql in sql_statements:
            try:
                stmt = conn.createStatement()
                stmt.execute(sql)
                results.append({"sql": sql, "status": "success", "error": None})
            except Exception as ex_stmt:
                results.append({"sql": sql, "status": "failed", "error": str(ex_stmt)})
            finally:
                try:
                    stmt.close()
                except Exception:
                    # Log or ignore close errors
                    pass

    except Exception as ex_conn:
        # If connection establishment fails
        err_msg = f"Connection setup failed: {ex_conn}"
        # Append an entry for each statement? Or a single global error?
        results.append({"sql": None, "status": "failed", "error": err_msg})
    finally:
        if conn:
            try:
                conn.close()
            except Exception:
                # Log or ignore
                pass

    return results

In [ ]:
def execute_sql_statements(
    statements: list[str],
    jdbc_url: str,
    access_token: str | None = None
) -> list[dict]:
    """
    Execute each SQL statement via JDBC using token authentication.
    Returns a list of dicts with:
      - 'sql': the SQL statement
      - 'status': 'success' or 'failed'
      - 'start', 'end': timestamps (UTC) when execution began & ended
      - 'error': error message when failed, else None
    """
    if access_token is None:
        access_token = mssparkutils.credentials.getToken("pbi")

    conn = None
    stmt = None
    results = []
    try:
        # Prepare JDBC properties for token authentication
        props = spark._sc._gateway.jvm.java.util.Properties()
        props.setProperty("accessToken", access_token)
        props.setProperty("encrypt", "true")

        driver_manager = spark._sc._gateway.jvm.java.sql.DriverManager
        conn = driver_manager.getConnection(jdbc_url, props)
        stmt = conn.createStatement()

        for sql in statements:
            try:
                print(f"[execute_sql_statements] Executing: {sql}")
                ts_start = datetime.utcnow()
                stmt.execute(sql)
                ts_end = datetime.utcnow()
                results.append({
                    "sql": sql,
                    "status": "success",
                    "start": ts_start,
                    "end": ts_end,
                    "error": None
                })
            except Exception as ex_stmt:
                print(f"[execute_sql_statements] Failed: {sql}\nException: {ex_stmt}")
                results.append({
                    "sql": sql,
                    "status": "failed",
                    "start": None,
                    "end": None,
                    "error": str(ex_stmt)
                })
                # continue to next statement
    except Exception as ex_conn:
        # Connection-level failure
        raise RuntimeError(f"Failed to open JDBC connection or statement: {ex_conn}") from ex_conn
    finally:
        # Clean up resources
        try:
            if stmt:
                stmt.close()
        except Exception as close_ex:
            print(f"[execute_sql_statements] Warning: could not close stmt: {close_ex}")
        try:
            if conn:
                conn.close()
        except Exception as close_ex2:
            print(f"[execute_sql_statements] Warning: could not close connection: {close_ex2}")

    return results